# Tiny GRU ONNX User Test

로컬 `model.onnx`를 직접 호출해서 두 개의 1인 사용자 프로필에 대한 next POI 추천을 확인합니다.

- API key는 `.env`의 `DATA_OPENAPI_KEY`에서 읽고 값은 출력하지 않습니다.
- 추천된 `contentId`는 TourAPI `KorService2/detailCommon2`로 관광지명(`title`)을 조회합니다.
- 첨부 문서는 공통 호출 규칙 참고용이며, contentId 상세 조회는 `KorService2/detailCommon2`를 사용합니다.

In [1]:
from __future__ import annotations

import json
import pickle
import socket
import time
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import onnxruntime as ort
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARTIFACT_DIR = ROOT / "artifacts" / "tiny_gru_onnx_experiment"
MODEL_PATH = ARTIFACT_DIR / "model.onnx"
FEATURE_ENCODER_PATH = ARTIFACT_DIR / "feature_encoder.pkl"
VOCAB_PATH = ARTIFACT_DIR / "content_id_vocab.json"
TRAIN_CONFIG_PATH = ARTIFACT_DIR / "train_config.json"
INPUT_PATH = ROOT / "data" / "processed" / "total_input.csv"
SEQ_PATH = ROOT / "data" / "processed" / "total_travel_seq_with_contentid.csv"

TOURAPI_DETAIL_COMMON_URL = "https://apis.data.go.kr/B551011/KorService2/detailCommon2"
MOBILE_APP = "kor_travel_recommendation"
TOP_K = 10

for path in [MODEL_PATH, FEATURE_ENCODER_PATH, VOCAB_PATH, TRAIN_CONFIG_PATH, INPUT_PATH, SEQ_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("ROOT:", ROOT)
print("ONNX model:", MODEL_PATH.relative_to(ROOT))

ROOT: S:\Study\AI
ONNX model: artifacts\tiny_gru_onnx_experiment\model.onnx


## Load API Key And Runtime Artifacts

In [2]:
def read_env_key(env_path: Path, key_name: str) -> str:
    if not env_path.exists():
        raise FileNotFoundError(f".env file not found: {env_path}")
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == key_name:
            return value.strip().strip('"').strip("'")
    raise RuntimeError(f"{key_name} was not found in {env_path}")


service_key = read_env_key(ROOT / ".env", "DATA_OPENAPI_KEY")
print("DATA_OPENAPI_KEY loaded:", bool(service_key))
print("DATA_OPENAPI_KEY value is hidden")

session = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])
with FEATURE_ENCODER_PATH.open("rb") as fp:
    feature_encoder = pickle.load(fp)
with VOCAB_PATH.open("r", encoding="utf-8") as fp:
    vocab = json.load(fp)
with TRAIN_CONFIG_PATH.open("r", encoding="utf-8") as fp:
    train_config = json.load(fp)

content_id_to_token = vocab["content_id_to_token"]
token_to_content_id = {int(key): value for key, value in vocab["token_to_content_id"].items()}
unk_token_id = content_id_to_token[vocab["unk_token"]]
max_sequence_len = int(train_config["max_sequence_len"])
feature_columns = list(feature_encoder.feature_names_in_)

print("ONNX inputs:", [(item.name, item.shape, item.type) for item in session.get_inputs()])
print("feature count:", len(feature_columns))
print("vocab size:", len(content_id_to_token))
print("max_sequence_len:", max_sequence_len)

DATA_OPENAPI_KEY loaded: True
DATA_OPENAPI_KEY value is hidden
ONNX inputs: [('user_features', ['batch', 1844], 'tensor(float)'), ('sequences', ['batch', 'sequence_length'], 'tensor(int64)'), ('lengths', ['batch'], 'tensor(int64)')]
feature count: 17
vocab size: 762
max_sequence_len: 20


## Build Seed Sequence And User Profiles

`USER_CONTENT_ID_SEQUENCE`를 직접 리스트로 지정하면 그 값을 사용합니다. `None`이면 기존 시퀀스 데이터에서 모델 vocab에 있는 contentId 3개를 자동으로 고릅니다.

In [3]:
USER_CONTENT_ID_SEQUENCE: list[str] | None = None


def normalize_content_id(value: Any) -> str:
    text = str(value).strip()
    return text[:-2] if text.endswith(".0") else text


def choose_seed_sequence(seq_path: Path, vocab_lookup: dict[str, int], n: int = 3) -> list[str]:
    seq_df = pd.read_csv(seq_path, encoding="utf-8-sig")
    matched = seq_df.loc[seq_df["CONTENT_ID"].notna()].copy()
    matched["CONTENT_ID_NORM"] = matched["CONTENT_ID"].map(normalize_content_id)
    matched = matched.sort_values(["travel_id", "day_index", "visit_order"], kind="mergesort")
    for _, group in matched.groupby("travel_id", sort=False):
        ids = [cid for cid in group["CONTENT_ID_NORM"].tolist() if cid in vocab_lookup]
        if len(ids) >= n:
            return ids[:n]
    return [cid for cid in vocab_lookup if not cid.startswith("<")][:n]


def clean_profile(row: pd.Series) -> dict[str, Any]:
    profile = {column: row[column] for column in feature_columns}
    for key, value in list(profile.items()):
        if pd.isna(value):
            profile[key] = np.nan
    return profile


def build_profiles(input_df: pd.DataFrame) -> list[dict[str, Any]]:
    female_candidates = input_df.loc[(input_df["p0_age"].eq(20)) & (input_df["p0_gender"].eq("여"))]
    if female_candidates.empty:
        raise RuntimeError("No 20s female template row found in total_input.csv")
    young_active_female = clean_profile(female_candidates.iloc[0])
    young_active_female.update(
        {
            "companion_count": 1,
            "has_child": 0,
            "has_elderly": 0,
            "has_disabled": 0,
            "p0_age": 20,
            "p0_gender": "여",
            "p1_age": np.nan,
            "p1_gender": np.nan,
            "p1_style": np.nan,
            "p1_home": np.nan,
            "p1_preferred": np.nan,
        }
    )

    male_candidates = input_df.loc[
        (input_df["p0_age"].eq(50)) & (input_df["p0_gender"].eq("남")) & (input_df["theme"].astype(str).eq("21"))
    ]
    if male_candidates.empty:
        raise RuntimeError("No 50s male theme=21 template row found in total_input.csv")
    heritage_male = clean_profile(male_candidates.iloc[0])
    heritage_male.update(
        {
            "theme": 21,
            "companion_count": 1,
            "has_child": 0,
            "has_elderly": 0,
            "has_disabled": 0,
            "p0_age": 50,
            "p0_gender": "남",
            "p1_age": np.nan,
            "p1_gender": np.nan,
            "p1_style": np.nan,
            "p1_home": np.nan,
            "p1_preferred": np.nan,
        }
    )

    return [
        {"profile_name": "20대 활발한 여자 1명", "features": young_active_female},
        {"profile_name": "50대 문화유산 선호 남자 1명", "features": heritage_male},
    ]


input_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
seed_content_ids = USER_CONTENT_ID_SEQUENCE or choose_seed_sequence(SEQ_PATH, content_id_to_token, n=3)
profiles = build_profiles(input_df)

print("seed_content_ids:", seed_content_ids)
display(pd.DataFrame([{**{"profile_name": item["profile_name"]}, **item["features"]} for item in profiles]))

seed_content_ids: ['2381373', '775394', '126508']


,profile_name,area_code,trip_days,theme,has_child,has_elderly,has_disabled,companion_count,p0_age,p0_gender,p0_style,p0_home,p0_preferred,p1_age,p1_gender,p1_style,p1_home,p1_preferred
0,20대 활발한 여자 1명,central,3,1,0,0,0,1,20,여,6;1;1;2;6;2;1;2,45,11110;30200;46820,NaN,NaN,NaN,NaN,NaN
1,50대 문화유산 선호 남자 1명,central,2,21,0,0,0,1,50,남,2;7;1;5;2;6;5;3,11,41830;41480;42780,NaN,NaN,NaN,NaN,NaN


## Run ONNX Recommendation

In [4]:
def encode_seed_sequence(content_ids: list[str]) -> tuple[np.ndarray, np.ndarray]:
    token_ids = [content_id_to_token.get(normalize_content_id(content_id), unk_token_id) for content_id in content_ids]
    token_ids = token_ids[-max_sequence_len:]
    if not token_ids:
        raise ValueError("seed contentId sequence is empty")
    return np.asarray([token_ids], dtype=np.int64), np.asarray([len(token_ids)], dtype=np.int64)


def recommend_for_profile(profile_name: str, features: dict[str, Any], top_k: int = TOP_K) -> pd.DataFrame:
    feature_df = pd.DataFrame([features], columns=feature_columns)
    user_features = feature_encoder.transform(feature_df).astype(np.float32)
    sequences, lengths = encode_seed_sequence(seed_content_ids)
    logits = session.run(
        None,
        {
            "user_features": user_features,
            "sequences": sequences,
            "lengths": lengths,
        },
    )[0]
    print(profile_name, "feature shape:", user_features.shape, "logits shape:", logits.shape)
    scores = logits[0]
    top_tokens = np.argsort(-scores)[:top_k]
    return pd.DataFrame(
        [
            {
                "profile_name": profile_name,
                "rank": rank,
                "contentId": token_to_content_id.get(int(token_id), str(token_id)),
                "token_id": int(token_id),
                "score": float(scores[token_id]),
            }
            for rank, token_id in enumerate(top_tokens, start=1)
        ]
    )


recommendation_df = pd.concat(
    [recommend_for_profile(item["profile_name"], item["features"], TOP_K) for item in profiles],
    ignore_index=True,
)
display(recommendation_df)

20대 활발한 여자 1명 feature shape: (1, 1844) logits shape: (1, 762)
50대 문화유산 선호 남자 1명 feature shape: (1, 1844) logits shape: (1, 762)


,profile_name,rank,contentId,token_id,score
0,20대 활발한 여자 1명,1,126078,26,1.595916
1,20대 활발한 여자 1명,2,1796079,192,1.490648
2,20대 활발한 여자 1명,3,741658,736,1.452070
3,20대 활발한 여자 1명,4,126081,29,1.451198
4,20대 활발한 여자 1명,5,2862152,493,1.299848
5,20대 활발한 여자 1명,6,130551,126,1.043624
6,20대 활발한 여자 1명,7,2779504,411,1.017737
7,20대 활발한 여자 1명,8,129785,108,0.863611
8,20대 활발한 여자 1명,9,4011128,691,0.685262
9,20대 활발한 여자 1명,10,1805965,193,0.684228


## Look Up TourAPI Titles

In [5]:
def looks_url_encoded(value: str) -> bool:
    return "%2" in value.lower()


def build_tourapi_url(base_url: str, params: dict[str, Any]) -> str:
    service_key = str(params["serviceKey"])
    other_params = {key: value for key, value in params.items() if key != "serviceKey"}
    service_key_param = f"serviceKey={service_key}" if looks_url_encoded(service_key) else urlencode({"serviceKey": service_key})
    return f"{base_url}?{service_key_param}&{urlencode(other_params)}"


def parse_tourapi_items(payload: dict[str, Any]) -> list[dict[str, Any]]:
    response = payload.get("response", {})
    body = response.get("body", {})
    items = body.get("items", {})
    raw_item = items.get("item", []) if isinstance(items, dict) else []
    if isinstance(raw_item, dict):
        raw_item = [raw_item]
    return [item for item in raw_item if isinstance(item, dict)] if isinstance(raw_item, list) else []


def fetch_detail_common(content_id: str, timeout: float = 10.0, retries: int = 1) -> dict[str, Any]:
    params = {
        "serviceKey": service_key,
        "MobileOS": "ETC",
        "MobileApp": MOBILE_APP,
        "_type": "json",
        "contentId": normalize_content_id(content_id),
        "numOfRows": 10,
        "pageNo": 1,
    }
    url = build_tourapi_url(TOURAPI_DETAIL_COMMON_URL, params)
    last_error = None
    for attempt in range(retries + 1):
        try:
            with urlopen(url, timeout=timeout) as response:
                payload = json.loads(response.read().decode("utf-8"))
            header = payload.get("response", {}).get("header", {})
            result_code = str(header.get("resultCode", ""))
            result_msg = str(header.get("resultMsg", ""))
            if result_code and result_code != "0000":
                return {
                    "contentId": normalize_content_id(content_id),
                    "title": None,
                    "addr1": None,
                    "contenttypeid": None,
                    "api_status": result_msg or f"resultCode={result_code}",
                }
            items = parse_tourapi_items(payload)
            if not items:
                return {
                    "contentId": normalize_content_id(content_id),
                    "title": None,
                    "addr1": None,
                    "contenttypeid": None,
                    "api_status": "NODATA_ERROR",
                }
            item = items[0]
            return {
                "contentId": normalize_content_id(content_id),
                "title": item.get("title"),
                "addr1": item.get("addr1"),
                "contenttypeid": item.get("contenttypeid"),
                "api_status": "ok",
            }
        except (HTTPError, URLError, TimeoutError, socket.timeout, OSError, json.JSONDecodeError) as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < retries:
                time.sleep(0.5 * (attempt + 1))
                continue
    return {
        "contentId": normalize_content_id(content_id),
        "title": None,
        "addr1": None,
        "contenttypeid": None,
        "api_status": last_error or "api_error",
    }


unique_content_ids = recommendation_df["contentId"].drop_duplicates().tolist()
detail_df = pd.DataFrame([fetch_detail_common(content_id) for content_id in unique_content_ids])
final_df = recommendation_df.merge(detail_df, on="contentId", how="left")
final_df = final_df[["profile_name", "rank", "contentId", "score", "title", "addr1", "contenttypeid", "api_status"]]
display(final_df)

,profile_name,rank,contentId,score,title,addr1,contenttypeid,api_status
0,20대 활발한 여자 1명,1,126078,1.595916,광안리해수욕장,부산광역시 수영구 광안해변로 219 (광안동),12,ok
1,20대 활발한 여자 1명,2,1796079,1.490648,성심당,대전광역시 중구 대종로480번길 15,39,ok
2,20대 활발한 여자 1명,3,741658,1.452070,한밭수목원,대전광역시 서구 둔산대로 169,12,ok
3,20대 활발한 여자 1명,4,126081,1.451198,해운대해수욕장,부산광역시 해운대구 해운대해변로 264,12,ok
4,20대 활발한 여자 1명,5,2862152,1.299848,밀락더마켓,부산광역시 수영구 민락수변로17번길 56,12,ok
5,20대 활발한 여자 1명,6,130551,1.043624,대전시립미술관,대전광역시 서구 둔산대로 155,14,ok
6,20대 활발한 여자 1명,7,2779504,1.017737,한빛탑,대전광역시 유성구 대덕대로 480,12,ok
7,20대 활발한 여자 1명,8,129785,0.863611,국립중앙과학관,대전광역시 유성구 대덕대로 481 (구성동),14,ok
8,20대 활발한 여자 1명,9,4011128,0.685262,KT&G 상상마당 부산,부산광역시 부산진구 서면로 39 (부전동),38,ok
9,20대 활발한 여자 1명,10,1805965,0.684228,전포카페거리,부산광역시 부산진구 동천로 92 (전포동),12,ok
